## Imports openbb

In [5]:
# 🔧 ENSURE PRODUCTION DATABASE IS USED (NOT TEST DB)
import os

# Set environment variable to use production database
os.environ["FMP_CACHE_TEST_MODE"] = "false"

print("✅ Environment configured:")
print(f"   FMP_CACHE_TEST_MODE = {os.environ.get('FMP_CACHE_TEST_MODE', 'not set')}")
print(f"   Database: openbb_fmp_cache (PRODUCTION)")
print(f"   NOT using test database (openbb_fmp_cache_test)")

# Import required libraries
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from openbb import obb

# Set pandas display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.precision', 2)

print("\n✅ Libraries imported successfully!")
print(f"   OpenBB Version: {obb.__version__ if hasattr(obb, '__version__') else 'N/A'}")


✅ Environment configured:
   FMP_CACHE_TEST_MODE = false
   Database: openbb_fmp_cache (PRODUCTION)
   NOT using test database (openbb_fmp_cache_test)

✅ Libraries imported successfully!
   OpenBB Version: N/A


In [6]:
# 🔧 FIX: FMP Cached Provider Charting

# The issue is that fmp_cached might return data in a slightly different format
# or with missing metadata that the charting system expects.
symbol = "AAPL"

try:
    # SOLUTION 1: Normalize the data before charting
    print(f"📊 Fetching data for {symbol} using fmp_cached provider...")
    res_cached = obb.equity.price.historical(
        symbol=symbol,
        provider="fmp_cached"
    )
    
    # Convert to DataFrame and back to ensure proper format
    df_normalized = res_cached.to_df()
    
    print(f"✅ Successfully loaded {len(df_normalized)} rows of data")
    print(f"   Columns: {list(df_normalized.columns)}")
    print(f"   Date range: {df_normalized.index[0]} to {df_normalized.index[-1]}")
    
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {str(e)}")
    print("\n💡 Troubleshooting:")
    print("   1. Make sure MySQL database is running")
    print("   2. Verify fmp_cached provider is installed")
    print("   3. Check database connection settings")
    import traceback
    traceback.print_exc()


📊 Fetching data for AAPL using fmp_cached provider...
Creating 67 flattened database tables...
✅ Created flattened table: analyst_estimates
✅ Created flattened table: available_indices
✅ Created flattened table: balance_sheet
✅ Created flattened table: balance_sheet_growth
✅ Created flattened table: calendar_dividend
✅ Created flattened table: calendar_earnings
✅ Created flattened table: calendar_events
✅ Created flattened table: calendar_ipo
✅ Created flattened table: calendar_splits
✅ Created flattened table: cash_flow
✅ Created flattened table: cash_flow_growth
✅ Created flattened table: company_filings
✅ Created flattened table: company_news
✅ Created flattened table: crypto_historical
✅ Created flattened table: crypto_search
✅ Created flattened table: currency_historical
✅ Created flattened table: currency_pairs
✅ Created flattened table: currency_snapshots
✅ Created flattened table: discovery_filings
✅ Created flattened table: earnings_call_transcript
✅ Created flattened table: e

Still have 9 gaps after caching for AAPL


✅ Successfully loaded 357 rows of data
   Columns: ['open', 'high', 'low', 'close', 'volume', 'vwap', 'change', 'change_percent', 'symbol', '_is_filled', '_fill_source_date', '_fill_type']
   Date range: 2024-11-29 to 2025-11-28


In [7]:
df_normalized

,open,high,low,close,volume,vwap,change,change_percent,symbol,_is_filled,_fill_source_date,_fill_type
date,,,,,,,,,,,,
2024-11-29,234.81,237.81,233.97,237.33,28481400,235.98,2.53,1.07e-02,AAPL,NaN,NaN,NaN
2024-11-30,234.81,237.81,233.97,237.33,0,235.98,0.00,NaN,AAPL,True,2024-11-29,previous_close
2024-12-01,234.81,237.81,233.97,237.33,0,235.98,0.00,NaN,AAPL,True,2024-11-30,previous_close
2024-12-02,237.27,240.79,237.16,239.59,48137103,238.70,2.32,9.78e-03,AAPL,NaN,NaN,NaN
2024-12-03,239.81,242.76,238.90,242.65,38861017,241.03,2.84,1.18e-02,AAPL,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-23,265.95,273.33,265.67,271.49,0,269.11,0.00,NaN,AAPL,True,2025-11-22,previous_close
2025-11-24,270.90,277.00,270.90,275.92,65585800,273.68,5.02,1.85e-02,AAPL,NaN,NaN,NaN
2025-11-25,275.27,280.38,275.25,276.97,46914220,276.97,1.70,6.18e-03,AAPL,NaN,NaN,NaN


In [10]:
# Create a simple candlestick chart with plotly directly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplot figure with volume
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=(f'{symbol} Stock Price', 'Volume'),
    row_heights=[0.7, 0.3]
)

# Add candlestick chart
fig.add_trace(
    go.Candlestick(
        x=df_normalized.index,
        open=df_normalized['open'],
        high=df_normalized['high'],
        low=df_normalized['low'],
        close=df_normalized['close'],
        name='OHLC'
    ),
    row=1, col=1
)

# Add volume bars
fig.add_trace(
    go.Bar(
        x=df_normalized.index,
        y=df_normalized['volume'],
        name='Volume',
        marker_color='lightblue'
    ),
    row=2, col=1
)

# Add SMA indicators if available
if 'close' in df_normalized.columns:
    # Calculate SMAs
    df_normalized['SMA_20'] = df_normalized['close'].rolling(window=20).mean()
    df_normalized['SMA_50'] = df_normalized['close'].rolling(window=50).mean()
    
    # Add SMA lines
    fig.add_trace(
        go.Scatter(
            x=df_normalized.index,
            y=df_normalized['SMA_20'],
            name='SMA 20',
            line=dict(color='orange', width=1)
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=df_normalized.index,
            y=df_normalized['SMA_50'],
            name='SMA 50',
            line=dict(color='blue', width=1)
        ),
        row=1, col=1
    )

# Update layout
fig.update_layout(
    title=f'{symbol} Price Chart with Volume',
    yaxis_title='Price',
    yaxis2_title='Volume',
    xaxis_rangeslider_visible=False,
    height=800,
    showlegend=True,
    hovermode='x unified'
)

# Display the chart
fig.show()

print(f"✅ Chart created successfully with {len(df_normalized)} data points")
print(f"📊 Date range: {df_normalized.index[0]} to {df_normalized.index[-1]}")


✅ Chart created successfully with 357 data points
📊 Date range: 2024-11-29 00:00:00 to 2025-11-28 00:00:00
